# Research Paper RAG Chatbot (Local Vector Store)

This notebook walks through loading a research paper PDF, chunking it into overlapping passages, indexing embeddings locally with NumPy cosine similarity, and running conversational Q&A with page citations.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the import path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_settings
from src.loader import load_pdf_chunks
from src.vectorstore import LocalVectorStore
from src.rag import ResearchAssistant

## 1. Load Configuration & PDF Chunks

Make sure `GEMINI_API_KEY` is configured in your `.env` file at the project root.

In [ ]:
settings = get_settings()
pdf_path = PROJECT_ROOT / "data" / "RAGPaper.pdf"

chunks = load_pdf_chunks(
    pdf_path,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)
print(f"Extracted {len(chunks)} chunks from {pdf_path.name}")
print("Sample chunk:", chunks[0].chunk_id, "| page", chunks[0].page)

## 2. Build or Load the Local Vector Index

In [ ]:
store = LocalVectorStore(PROJECT_ROOT / settings.vector_db_dir, index_name="ragpaper")
assistant = ResearchAssistant(store=store, settings=settings)

if store.exists():
    store.load()
    print(f"Loaded cached index ({len(store.chunks)} chunks)")
else:
    assistant.index_chunks(chunks)
    print(f"Indexed {len(chunks)} chunks locally")

## 3. Ask Questions & Follow-ups

In [ ]:
answer, matches = assistant.ask("What is Retrieval-Augmented Generation and how does it combine parametric and non-parametric memory?")
print(answer)
print("\nCited pages:", sorted({c.page for c, _ in matches}))

In [ ]:
# Follow-up question using conversation memory
followup_answer, followup_matches = assistant.ask("What benchmarks did the authors evaluate it on?")
print(followup_answer)
print("\nCited pages:", sorted({c.page for c, _ in followup_matches}))